In [2]:
import numpy as np
from safetensors.numpy import load_file
from transformers import AutoTokenizer

/Users/alex/Documents/programming/repos/transformers-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [4]:
class GELU:
    def __call__(self, x):
        return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

    def __str__(self):
        return "GELU"

In [5]:
class Linear:
    def __init__(self, weights, bias) -> None:
        self.weights = weights
        self.bias = bias

    def __call__(self, x):
        return x @ self.weights + self.bias

    def __str__(self):
        return f"Linear weights: {self.weights.shape}, bias: {self.bias.shape}"

In [6]:
class LayerNorm:
    def __init__(self, weight, bias, eps=1e-5):
        self.weight = weight
        self.bias = bias
        self.eps = eps

    def __call__(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)

        x_norm = (x - mean) / np.sqrt(var + self.eps)

        return self.weight * x_norm + self.bias

    def __str__(self):
        return f"LayerNorm weights: {self.weight.shape}, bias: {self.bias.shape}"

In [7]:
class Attention:
    def __init__(self, attn_weights, attn_bias, proj_weights, proj_bias) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias

    def __call__(self, x):
        qkv = x @ self.attn_weights + self.attn_bias

        q, k, v = np.split(qkv, 3, axis=-1)

        scores = q @ k.T
        scores = scores / np.sqrt(k.shape[-1])
        scores = softmax(scores)

        out = scores @ v

        out = out @ self.proj_weights + self.proj_bias

        return out

    def __str__(self):
        return f"Attention weights: {self.attn_weights.shape}, bias: {self.attn_bias.shape}, proj weights: {self.proj_weights.shape}, proj bias: {self.proj_bias.shape}"

In [8]:
class MultiHeadAttention:
    def __init__(
        self, attn_weights, attn_bias, proj_weights, proj_bias, num_heads=8
    ) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias
        self.num_heads = num_heads

    def __call__(self, x):
        seq_len = x.shape[0]
        hidden_dim = x.shape[1]  # Should be 512
        head_dim = hidden_dim // self.num_heads  # 512 // 8 = 64

        # 1. Project input to Q, K, V
        qkv = x @ self.attn_weights + self.attn_bias  # (seq_len, 512*3)
        q, k, v = np.split(qkv, 3, axis=-1)  # Each: (seq_len, 512)

        # 2. Split into multiple heads
        # Reshape from (seq_len, 512) to (seq_len, num_heads, head_dim)
        q = q.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        k = k.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        v = v.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)

        # 3. Transpose to (num_heads, seq_len, head_dim) for batch processing
        q = q.transpose(1, 0, 2)  # (8, seq_len, 64)
        k = k.transpose(1, 0, 2)  # (8, seq_len, 64)
        v = v.transpose(1, 0, 2)  # (8, seq_len, 64)

        # 4. Compute attention scores for all heads
        scores = q @ k.transpose(0, 2, 1)  # (8, seq_len, seq_len)
        scores = scores / np.sqrt(head_dim)  # Scale by sqrt(64)

        # 5. Apply causal mask (prevent attending to future tokens)
        causal_mask = np.tril(np.ones((seq_len, seq_len)))  # Lower triangular matrix
        scores = np.where(causal_mask == 0, -1e10, scores)  # Mask future positions

        # 6. Apply softmax to get attention weights
        attn_weights = softmax(scores, axis=-1)  # (8, seq_len, seq_len)

        # 7. Apply attention weights to values
        output = attn_weights @ v  # (8, seq_len, 64)

        # 8. Concatenate heads back together
        output = output.transpose(1, 0, 2)  # (seq_len, 8, 64)
        output = output.reshape(seq_len, hidden_dim)  # (seq_len, 512)

        # 9. Final projection
        output = output @ self.proj_weights + self.proj_bias

        return output

    def __str__(self):
        return f"MultiHeadAttention weights: {self.attn_weights.shape}, bias: {self.attn_bias.shape}, proj weights: {self.proj_weights.shape}, proj bias: {self.proj_bias.shape}"

In [9]:
class Transformer:
    def __init__(self, model) -> None:
        self.model = model
        self.attention_blocks = []
        self.ffn_blocks = []

        for i in range(4):
            # Attention block (LayerNorm + Attention)
            ln_1_weight = self.model[f"transformer.h.{i}.ln_1.weight"]
            ln_1_bias = self.model[f"transformer.h.{i}.ln_1.bias"]
            attn_c_attn_weight = self.model[f"transformer.h.{i}.attn.c_attn.weight"]
            attn_c_attn_bias = self.model[f"transformer.h.{i}.attn.c_attn.bias"]
            attn_c_proj_weight = self.model[f"transformer.h.{i}.attn.c_proj.weight"]
            attn_c_proj_bias = self.model[f"transformer.h.{i}.attn.c_proj.bias"]

            self.attention_blocks.append(
                {
                    "ln": LayerNorm(ln_1_weight, ln_1_bias),
                    "attn": MultiHeadAttention(
                        attn_c_attn_weight,
                        attn_c_attn_bias,
                        attn_c_proj_weight,
                        attn_c_proj_bias,
                    ),
                }
            )

            # FFN block (LayerNorm + Linear + GELU + Linear)
            ln_2_weight = self.model[f"transformer.h.{i}.ln_2.weight"]
            ln_2_bias = self.model[f"transformer.h.{i}.ln_2.bias"]
            ffn_1_weight = self.model[f"transformer.h.{i}.mlp.c_fc.weight"]
            ffn_1_bias = self.model[f"transformer.h.{i}.mlp.c_fc.bias"]
            ffn_2_weight = self.model[f"transformer.h.{i}.mlp.c_proj.weight"]
            ffn_2_bias = self.model[f"transformer.h.{i}.mlp.c_proj.bias"]

            self.ffn_blocks.append(
                {
                    "ln": LayerNorm(ln_2_weight, ln_2_bias),
                    "linear1": Linear(ffn_1_weight, ffn_1_bias),
                    "gelu": GELU(),
                    "linear2": Linear(ffn_2_weight, ffn_2_bias),
                }
            )

    def __call__(self, x):
        for attn_block, ffn_block in zip(self.attention_blocks, self.ffn_blocks):
            # Attention with residual
            attn_out = attn_block["attn"](attn_block["ln"](x))
            x += attn_out  # ← Residual connection

            # FFN with residual
            ffn_out = ffn_block["linear2"](
                ffn_block["gelu"](ffn_block["linear1"](ffn_block["ln"](x)))
            )
            x += ffn_out  # ← Residual connection

        # final layer norm
        x = LayerNorm(
            self.model["transformer.ln_f.weight"], self.model["transformer.ln_f.bias"]
        )(x)

        logits = x @ self.model["transformer.wte.weight"].T
        token_ids = np.argmax(logits, axis=-1)

        return int(token_ids[-1])

In [10]:
from abc import ABC, abstractmethod


class Model(ABC):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    @abstractmethod
    def next_token(self, input_text: str) -> str:
        pass

    def generate(self, input_text: str, max_tokens: int = 100) -> str:
        text = input_text
        eos_token = getattr(self.tokenizer, "eos_token", None)

        for _ in range(max_tokens):
            next_token = self.next_token(text)
            text += next_token
            if eos_token and next_token == eos_token:
                break

        return text

In [11]:
class GPT2(Model):
    def __init__(self) -> None:
        tokenizer = AutoTokenizer.from_pretrained(
            "erwanf/gpt2-mini", cache_dir="./data/hf"
        )
        super().__init__(tokenizer)
        self.model_weights = load_file("./data/model.safetensors")
        self.model = Transformer(self.model_weights)

    def next_token(self, input_text: str) -> str:
        input_tokens = self.tokenizer.encode(input_text)
        token_embeddings = self.model_weights["transformer.wte.weight"][input_tokens]
        pos_embeddings = self.model_weights["transformer.wpe.weight"][
            : len(input_tokens)
        ]
        embeddings = token_embeddings + pos_embeddings
        next_token_id = self.model(embeddings)
        return self.tokenizer.decode(next_token_id)

In [13]:
text = "Anthropic is a company"

gpt2 = GPT2()
result = gpt2.generate(text, max_tokens=50)
print(result)

Anthropic is a company that has been working on the project for over a decade.



quer.





quer.









ument.

ument.

ument.

ument.
